# Marso submission — colour-aware Diffusion Policy (Kaggle)

Runs the whole pipeline on **Kaggle's free GPU** (separate quota from Colab; data already mounted).

## Before you run — three settings (right sidebar, click the **...** / Settings)
1. **Accelerator -> GPU T4 x2** (or P100). Without this, ManiSkill won't run.
2. **Internet -> On.** Needed for `pip install` and cloning the fork.
3. **Add Input -> Competitions ->** search *marso-hack-berlin-2026* -> **Add**. This mounts the
   demos under `/kaggle/input/` (no Kaggle token needed). You must have joined the competition
   (accepted the rules) first.

Then **Run All**. Training on `easy` takes ~1.5 h. When it's done you **download one file**
(the checkpoint) and send me the eval numbers — I commit + push it from VS Code.


## 1. Clone the fork + install

In [ ]:
import os
%cd /kaggle/working
if not os.path.exists('/kaggle/working/berlin-marso-hackathon'):
    !git clone https://github.com/patcadragos/berlin-marso-hackathon.git
%cd /kaggle/working/berlin-marso-hackathon

!pip install mani-skill==3.0.1 diffusers==0.38.0 gymnasium torch torchvision hydra-core -q
!pip install -e . -q

# Headless rendering (offscreen Vulkan/EGL)
os.environ['DISPLAY'] = ''
os.environ['PYOPENGL_PLATFORM'] = 'egl'


## 2. Stage the competition demos

The data is already mounted under `/kaggle/input/` (because you added it as Input above), so this
just copies it into place — **no Kaggle token needed**.


In [ ]:
!python il/download_demos.py
!ls -la il/demos/*/ 2>/dev/null | head -30


## 3. Train (easy first — the reliable, bankable score)

Easy = 2 parcels, fixed positions: behavior cloning can actually reproduce the demos and score
points. (Hard — 6 parcels, bin-swaps, 200-step cap — repeatedly scored 0; that's the IL wall,
not a bug.) Colour-aware encoder + augmentation are on. ~1.5 h on a T4.


In [ ]:
!python il/train.py method=dp_rgb_color demo_dir=easy flags.total_iters=20000 flags.eval_freq=4000


## 4. Evaluate on all three levels

Prints `sort_accuracy` per level and the weighted `final_score = 0.2*easy + 0.3*medium + 0.5*hard`.


In [ ]:
import subprocess, re, os

CKPT = "il/baselines/diffusion_policy/runs/warehouse_rgb_dp_color/checkpoints/best_eval_sort_accuracy.pt"
assert os.path.exists(CKPT), "checkpoint not found -- did training finish? " + CKPT

scores = {}
for level in ["easy", "medium", "hard"]:
    cmd = [
        "python", "eval.py", "difficulty=" + level,
        "policy=warehouse_sort.il_policy:load_dp_rgb_color",
        "checkpoint=" + CKPT, "eval_config=conf/eval/default.yaml",
    ]
    print("=== evaluating " + level + " ===")
    out = subprocess.run(cmd, capture_output=True, text=True)
    print(out.stdout[-3000:])
    if out.returncode != 0:
        print(out.stderr[-3000:])
    m = re.search(r"sort.?accuracy[:=]\s*([0-9.]+)", out.stdout, re.IGNORECASE)
    scores[level] = float(m.group(1)) if m else None

print("scores:", scores)
if all(v is not None for v in scores.values()):
    final = 0.2 * scores["easy"] + 0.3 * scores["medium"] + 0.5 * scores["hard"]
    print("final_score =", round(final, 4))
else:
    print("Could not auto-parse one or more sort_accuracy lines -- read them from the stdout above.")


## 5. Download the checkpoint

This copies the trained checkpoint to `/kaggle/working/` so you can download it: open the
**Output** panel (right sidebar) or the file browser, find `best_eval_sort_accuracy.pt`, and
download it to your PC. Then tell me the scores from step 4 and I'll commit + push it from VS Code.


In [ ]:
import shutil, os
src = "il/baselines/diffusion_policy/runs/warehouse_rgb_dp_color/checkpoints/best_eval_sort_accuracy.pt"
dst = "/kaggle/working/best_eval_sort_accuracy.pt"
shutil.copy(src, dst)
print("ready to download:", dst, "(%.1f MB)" % (os.path.getsize(dst) / 1e6))

# also surface the best eval rollout video, if any
import glob
vids = sorted(glob.glob("il/baselines/diffusion_policy/runs/*/videos/*.mp4")
              + glob.glob("outputs/**/videos/*.mp4", recursive=True), key=os.path.getmtime)
if vids:
    shutil.copy(vids[-1], "/kaggle/working/eval_rollout.mp4")
    print("rollout video ready to download: /kaggle/working/eval_rollout.mp4")
